# Project 2 - NCSU ST 554
### Author: Trevor Lillywhite
### Due Date: March 24, 2026

## Part I - Creating a Class

**Objective:** Create our own class called `SparkDataCheck` that works on Spark SQL style data frames. 

This will be created in a `.py` file and tested here in a pySpark3 notebook.

The `SparkDataCheck` class can be used to perform various checks on a spark DataFrame. 

*Instantiation:*
+ The default instantiation is to create an object by directly supplying a spark dataframe. 
+ There are also two class methods to alternatively instantiate an object using a csv file (`.from_csv()`) or a `pandas` DataFrame (`.from_pandas()`).

*Validation Methods:* There are three validation methods.
+ `.check_within_limits()` checks if each value in a numeric column is within user defined limits (inclusive). It returns a dataframe with an appended column of Boolean values.
+ `.check_string()` checks if each value in a string column falls within a user specified set of levels and returns the dataframe with an appended column of Boolean values.
+ `.check_Null()` checks if each value in a column is missing (`NULL`) and returns a dataframe with an appended column of Boolean values.

*Summarization Methods:* There are two summarization methods.
+ `.find_minmax()` reports the min and max of a numeric column specified by the user as a column name. There is an optional grouping variable. If no grouping variable is supplied, the min and max are across the entire column. If a grouping variable is supplied, the min and max of each group is reported. If no column is specified, then all numeric columns are reported (including grouping, if specified). 
+ `.find_counts` reports the counts associated with one or two string columns. The first string column is required and the second is optional. 

Below, the class will be tested to verify functionality. The class, which was defined in a `.py` file, will be imported in this notebook for testing. 

#### Import script

This will give access to the class I created

In [2]:
# Temporarily change working directory
%cd Project_2          # Note: if directory is already changed, this will return a non-fatal error

# Import relevant modules
from pyspark.sql import SparkSession
import pandas as pd
from pyspark.sql import functions as F
import pyspark.pandas as ps

# Import script
import project2_script

[Errno 2] No such file or directory: 'Project_2 # Note: if directory is already changed, this will return a non-fatal error'
/home/jupyter-tflillyw@ncsu.edu/Project_2


In [3]:
# Add cell to reload module (supports troubleshooting)
import importlib
importlib.reload(project2_script)

<module 'project2_script' from '/home/jupyter-tflillyw@ncsu.edu/Project_2/project2_script.py'>

result_via_pandas = project2_script.SparkDataCheck.from_pandas(test_session, p_df)
result_via_pandas

result_via_pandas.df.show()

result_via_csv = project2_script.SparkDataCheck.from_csv(test_session, 'weekly_nfl_data.csv')
result_via_csv

result_via_csv.check_within_limits('position', upper=1)

check3 = result_via_csv.check_Null('player_name')
check3.df.select('player_name', 'player_name_is_Null').orderBy(F.rand()).show(10)

check4 = result_via_csv.find_minmax(column = 'rushing_yards', group = 'season')
check4

check5 = result_via_csv.find_minmax(column='season')
check5

check6 = result_via_csv.find_counts(column1 = 'position', 
                                    column2 = 'opponent_team')
check6

In [ ]:
#   END TROUBLESHOOTING

### Read in air quality data

The air quality dataset will be used to test the class's functionality. 

This was used in the first project and is available here: https://www4.stat.ncsu.edu/online/datasets/air.csv

We will use the custom-defined method to create an instance of the class from the csv file downloaded from the website and uploaded to the JupyterHub environment in the same folder as the `.py` script and this notebook.

To do that, we first need to create a spark session.

In [5]:
# Create test session for the air quality data
air_session = SparkSession.builder.appName('my_app')\
                    .config("spark.sql.ansi.enabled", "false")\
                    .getOrCreate()

Now, we can create our class instance.

In [7]:
air_data = project2_script.SparkDataCheck.from_csv(
    air_session, 'air.csv')
air_data.df.show(1)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


26/03/22 20:18:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-tflillyw@ncsu.edu/Project_2/air.csv


There is a warning above because the csv file didn't include a name for the first column. The program was still able to run, as it created a name for that column (`_c0`). 

Before we test our various methods on the class instance, we need to do a small amount of data cleaning. First, we will clean the column names so they don't include periods, which could be misinterpreted. Additionally, any missing values are encoded as the value "-200", which will skew our numerical statistics. Further, since Date is the only string column, we will cast the Time column to be a string so we can better test the associated method. We will also name the first column of the csv to prevent the warning from displaying again. 

In [8]:
# Clean dataset column names

# Rename first column (was unnamed in csv)
air_data.df = air_data.df.withColumnRenamed('_c0', 'observation')

# Rename other columns (remove decimals)
air_data.df = air_data.df.withColumnRenamed('PT08.S1(CO)', 'PT08_S1(CO)')
air_data.df = air_data.df.withColumnRenamed('PT08.S2(NMHC)', 'PT08_S2(NMHC)')
air_data.df = air_data.df.withColumnRenamed('PT08.S3(NOx)', 'PT08_S3(NOx)')
air_data.df = air_data.df.withColumnRenamed('PT08.S4(NO2)', 'PT08_S4(NO2)')
air_data.df = air_data.df.withColumnRenamed('PT08.S5(O3)', 'PT08_S5(O3)')

# Write to CSV to permanently change header names
air_data.df.write.csv("air_modified.csv", header=True, mode="overwrite")

26/03/22 20:18:30 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-tflillyw@ncsu.edu/Project_2/air.csv


In [9]:
# Read modified csv
air_data = project2_script.SparkDataCheck.from_csv(
    air_session, 'air_modified.csv')
air_data.df.show(1)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


In [10]:
air_data.df.describe().show()

26/03/22 20:18:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+--------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|summary|       observation|    Date|            CO(GT)|       PT08_S1(CO)|           NMHC(GT)|          C6H6(GT)|    PT08_S2(NMHC)|          NOx(GT)|      PT08_S3(NOx)|           NO2(GT)|      PT08_S4(NO2)|       PT08_S5(O3)|                T|               RH|                AH|
+-------+------------------+--------+------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+------------------+------------------+-----------------+-----------------+------------------+
|  count|              9357|    9357|              9357|              9357|               9357|              9357|             9357|             9357|    

In [11]:
# Clean dataset values
# Replace values of "-200" with `NULL`. 
air_data.df = air_data.df.replace(-200, None)
air_data.df.describe().show()

+-------+------------------+--------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+
|summary|       observation|    Date|            CO(GT)|       PT08_S1(CO)|          NMHC(GT)|         C6H6(GT)|     PT08_S2(NMHC)|          NOx(GT)|     PT08_S3(NOx)|           NO2(GT)|      PT08_S4(NO2)|       PT08_S5(O3)|                T|                RH|               AH|
+-------+------------------+--------+------------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+------------------+------------------+------------------+-----------------+------------------+-----------------+
|  count|              9357|    9357|              7674|              8991|               914|             8991|              8991|             7718|           

We successfully eliminated the -200 encoding of missing values. The only cleaning step remaining is to change the Time column to be a string.

In [12]:
# Cast Date and Time columns to be strings
air_data.df = air_data.df.withColumn("Time", F.col("Time").cast("string"))
air_data.df.printSchema()

root
 |-- observation: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08_S1(CO): integer (nullable = true)
 |-- NMHC(GT): integer (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08_S2(NMHC): integer (nullable = true)
 |-- NOx(GT): integer (nullable = true)
 |-- PT08_S3(NOx): integer (nullable = true)
 |-- NO2(GT): integer (nullable = true)
 |-- PT08_S4(NO2): integer (nullable = true)
 |-- PT08_S5(O3): integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



We are finally ready to test our class!

### Provide examples using each method

We will provide a few examples of using each of the methods on this object. We will do this robustly to show that error messages show when appropriate and when the method can run using varying amounts of arguments. 

In some cases, we will randomize the values returned, effectively shuffling the deck to let us see a broad cross-seciton of data (not limited by the behavior of the first several observations). This randomization will change every time the code is run.

#### Test `check_within_limits()` method

**Test 1:** Provide no limits (bounds).

**Expected behavior:** Error

In [13]:
test1 = air_data.check_within_limits('T')

Error: Must specify at least one upper or lower bound.


*Result:* Success!

**Test 2:** Provide both limits (bounds). Provide non-numeric column.

**Expected behavior:** Warning, but still return unmodified dataframe.

In [14]:
test2 = air_data.check_within_limits('Date', lower=0, upper=1000)
test2.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|        683|  4/8/2004|2026-03-22 05:00:00|   0.8|        869|      68|     2.8|          642|     69|        1291|     62|        1288|        723| 8.1|72.1|0.7798|
|       6955|12/25/2004|2026-03-22 13:00:00|   5.5|       1609|    NULL|    18.1|         1243|    602|         505|    178|        1491|       2100|11.4|62.9|0.8454|
|       2724|  7/2/2004|2026-03-22 06:00:00|   0.8|        993|    NULL|     5.6|          795|     60|         888|     63|        1629|        770|24.8|50.6|1.5654

*Result:* Success!

**Test 3:** Provide both limits (bounds). Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results.

In [15]:
test3 = air_data.check_within_limits('T', lower=20, upper=30)
test3.df.orderBy(F.rand()).show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       2104| 6/6/2004|2026-03-22 10:00:00|   0.9|        806|    NULL|     3.4|          675|     61|        1270|     61|        1414|        528|23.1|39.4|1.0983|           true|
|       2093| 6/5/2004|2026-03-22 23:00:00|   1.7|       1008|    NULL|     8.0|          900|     55|         907|     67|        1681|        850|21.6|53.6|1.3674|           true|
|        370|3/26/2004|2026-03-22 04:00:00|   0.7|        925|      40|     2.2|          

*Result:* Success!

**Test 4:** Provide lower limit only. Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. Range is interpreted as anything above the lower limit.

In [16]:
test4 = air_data.check_within_limits('T', lower=20)
test4.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       8244| 2/17/2005|2026-03-22 06:00:00|   0.5|        893|    NULL|     1.1|          507|     59|        1111|     51|         794|        489| 6.9|42.2|0.4224|          false|
|       7959|  2/5/2005|2026-03-22 09:00:00|   4.1|       1165|    NULL|    11.3|         1024|    759|         646|    310|        1134|       1365| 4.1|52.7|0.4345|          false|
|       8738|  3/9/2005|2026-03-22 20:00:00|   5.9|       1552|    NULL|    24.4|    

*Result:* Success!

**Test 5:** Provide upper limit only. Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. Range is interpreted as anything below the upper limit.

In [19]:
test5 = air_data.check_within_limits('T', upper=22)
test5.df.orderBy(F.rand()).show(10)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+
|       6928|12/24/2004|2026-03-22 10:00:00|   6.6|       1462|    NULL|    18.8|         1263|    750|         510|    170|        1421|       2082| 6.7|59.7| 0.589|           true|
|       7247|  1/6/2005|2026-03-22 17:00:00|   1.9|       1173|    NULL|     7.7|          885|    274|         685|    134|        1288|       1038|15.6|54.8| 0.966|           true|
|       4151| 8/30/2004|2026-03-22 17:00:00|  NULL|       1099|    NULL|    11.9|    

*Result:* Success!

#### Test `check_string()` method

**Test 6:** Provide invalid column name (not in DataFrame).

**Expected behavior:** Error.

In [20]:
test6 = air_data.check_string('bad_name', levels='wrong')

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 7:** Provide numeric column.

**Expected behavior:** Error.

In [21]:
test7 = air_data.check_string('T', levels='25')

*Result:* Success!

**Test 8:** Provide string column. Provide single "level".

**Expected behavior:** Successful execution. New column with Boolean results. (Note: no result randomization this time)

In [22]:
test8 = air_data.check_string('Date', levels='3/10/2004')
test8.df.show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|              true|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|              t

*Result:* Success!

**Test 9:** Provide string column. Provide multiple "levels".

**Expected behavior:** Successful execution. New column with Boolean results. (Note: no result randomization this time)

In [23]:
test9 = air_data.check_string('Date', levels=['3/10/2004', '3/12/2004'])
test9.df.show(35)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|              true|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|              t

*Result:* Success!

**Test 10:** Provide string column. Provide empty list of "levels".

**Expected behavior:** Successful execution. New column with Boolean results, all "false". (Note: no result randomization this time)

In [24]:
test10 = air_data.check_string('Date', levels=[])
test10.df.show(10)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|             fa

*Result:* Success!

#### Test `check_Null()` method

**Test 11:** Provide invalid column name (not in DataFrame).

**Expected behavior:** Error.

In [25]:
test11 = air_data.check_Null('bad_name')

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 12:** Provide column name that isn't a string.

**Expected behavior:** Error.

In [26]:
test12 = air_data.check_Null(column=12)

Error: Column name must be a string


*Result:* Success!

**Test 13:** Provide numeric column.

**Expected behavior:** Successful execution. New column with Boolean results. (Note: might need to be run several times to actually find a NULL result)

In [27]:
test13 = air_data.check_Null('NOx(GT)')
test13.df.show(25)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|          false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1

*Result:* Success!

**Test 14:** Provide string column.

**Expected behavior:** Successful execution. New column with Boolean results. (Note: Date and Time are never null, so this shouldn't ever produce a "true" Boolean result.)

In [28]:
test14 = air_data.check_Null('Date')
test14.df.show(35)

+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+
|observation|     Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|Date_is_Null|
+-----------+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+
|          0|3/10/2004|2026-03-22 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|             false|          false|       false|
|          1|3/10/2004|2026-03-22 19:00:00|   2.0|       1292|     112|     9.4|

*Result:* Success!

We also see the method "chaining" even though we aren't directly chaining in the same line of code because each time we call it, we modify the underlying dataframe inside the object. Similar results would be observed if we directly chained the methods in a single line of code.

#### Test `find_minmax()` method

**Test 15:** Provide bad column name (not in DataFrame)

**Expected behavior:** Error

In [29]:
test15 = air_data.find_minmax(column='football')
test15

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 16:** Provide non-numeric column.

**Expected behavior:** Error

In [30]:
test16 = air_data.find_minmax('Date')
test16

Error: Column Date is non-numeric. Provide a numeric column.


*Result:* Success!

**Test 17:** Provide numeric column. Do not provide a group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame simply reporting the min and max values for that column.

In [31]:
test17 = air_data.find_minmax(column='NOx(GT)')
test17

,Min,Max
NOx(GT),2,1479


*Result:* Success!

**Test 18:** Provide numeric column. Provide a group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for that column grouped by the 'group' argument.

In [32]:
test18 = air_data.find_minmax(column='NOx(GT)', group='Date')
test18

,NOx(GT)_min,NOx(GT)_max
Date,,
1/1/2005,145.0,832.0
1/10/2005,64.0,698.0
1/11/2005,96.0,662.0
1/12/2005,162.0,910.0
1/13/2005,161.0,760.0
...,...,...
9/5/2004,NaN,NaN
9/6/2004,NaN,NaN
9/7/2004,145.0,232.0


*Result:* Success!

**Test 19:** Do not provide a column or group.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for each numeric column with no grouping.

In [33]:
test19 = air_data.find_minmax()
test19

,min,max
observation,0.0000,9356.000
CO(GT),0.1000,11.900
PT08_S1(CO),647.0000,2040.000
NMHC(GT),7.0000,1189.000
C6H6(GT),0.1000,63.700
PT08_S2(NMHC),383.0000,2214.000
NOx(GT),2.0000,1479.000
PT08_S3(NOx),322.0000,2683.000
NO2(GT),2.0000,340.000
PT08_S4(NO2),551.0000,2775.000


*Result:* Success!

**Test 20:** Provide a group. Do not provide a column.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the min and max values for each numeric column, grouped by the 'group' argument.

In [34]:
test20 = air_data.find_minmax(group='Date')
test20

,observation_min,observation_max,CO(GT)_min,CO(GT)_max,PT08_S1(CO)_min,PT08_S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,PT08_S4(NO2)_min,PT08_S4(NO2)_max,PT08_S5(O3)_min,PT08_S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,,,,,,,,,,,,,,,,,,,,,
1/1/2005,7110,7133,1.0,4.7,915.0,1472.0,NaN,NaN,3.0,16.6,...,897.0,1344.0,879.0,1735.0,2.6,12.8,32.3,63.2,0.4375,0.5961
1/10/2005,7326,7349,0.4,3.5,958.0,1523.0,NaN,NaN,1.3,21.4,...,1122.0,1804.0,707.0,1634.0,12.3,14.4,63.6,73.3,0.9912,1.0567
1/11/2005,7350,7373,0.7,5.1,977.0,1526.0,NaN,NaN,2.1,26.5,...,1106.0,1855.0,830.0,1738.0,11.7,13.8,56.3,71.9,0.8754,1.0087
1/12/2005,7374,7397,1.0,5.9,1002.0,1526.0,NaN,NaN,4.2,28.4,...,1139.0,1902.0,958.0,1770.0,8.7,15.8,48.8,79.4,0.8428,0.9467
1/13/2005,7398,7421,0.9,5.2,891.0,1523.0,NaN,NaN,3.2,26.6,...,1054.0,1811.0,798.0,1783.0,6.2,14.2,54.1,82.1,0.7751,0.9166
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9/5/2004,4278,4301,0.6,5.8,784.0,1399.0,NaN,NaN,1.9,30.3,...,1060.0,2052.0,342.0,1652.0,25.4,34.5,17.5,38.6,0.8162,1.2477
9/6/2004,4302,4325,0.4,2.9,826.0,1133.0,NaN,NaN,1.0,13.5,...,1189.0,1669.0,385.0,1105.0,23.5,30.1,26.7,49.6,1.1217,1.4247
9/7/2004,4326,4349,0.6,4.1,847.0,1311.0,NaN,NaN,2.7,22.0,...,1258.0,1957.0,497.0,1486.0,21.9,32.5,19.4,48.3,0.9279,1.3688


*Result:* Success!

#### Test `find_counts()` method

**Test 21:** Provide a bad column name (not in the DataFrame).

**Expected behavior:** Error.

In [35]:
test21 = air_data.find_counts(column1='date')
test21

Error: Invalid column (not in DataFrame)


*Result:* Success!

**Test 22:** Provide a single column name for a numeric column.

**Expected behavior:** Error.

In [36]:
test22 = air_data.find_counts(column1='T')
test22

Error: Column T is numeric. Provide a string column.


*Result:* Success!

**Test 23:** Provide a numeric column and then a string column.

**Expected behavior:** Error.

In [37]:
test23 = air_data.find_counts(column1='RH', column2='Date')
test23

Error: Column RH is numeric. Provide a string column.


*Result:* Success!

**Test 24:** Provide a single string column.

**Expected behavior:** Successful execution. Results will be a `pandas` DataFrame reporting the counts for each level of that column's values.

In [38]:
test24 = air_data.find_counts(column1='Date')
test24

,Date,Date_count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,8/16/2004,24
387,12/20/2004,24
388,3/5/2005,24
389,4/4/2005,15


*Result:* Success!

**Test 25:** Provide two string columns.

**Expected behavior:** Successful execution. Results will be a pandas DataFrame reporting the counts for each level of each column's values.

In [39]:
test25 = air_data.find_counts(column1='Date', column2='Time')
test25

,Date,Date_count,Time,Time_count
0,9/2/2004,24,2026-03-22 23:00:00,390.0
1,12/26/2004,24,2026-03-22 06:00:00,390.0
2,2/18/2005,24,2026-03-22 00:00:00,390.0
3,10/10/2004,24,2026-03-22 20:00:00,390.0
4,10/11/2004,24,2026-03-22 22:00:00,390.0
...,...,...,...,...
386,8/16/2004,24,NaN,NaN
387,12/20/2004,24,NaN,NaN
388,3/5/2005,24,NaN,NaN
389,4/4/2005,15,NaN,NaN


*Result:* Success!

**Test 26:** Chain two validation methods. We will use untouched columns to prove that previous methods/results didn't affect this. 

**Expected behavior:** Successful execution. New Boolean columns will be added for 'NMHC(GT)_is_Null' and 'PT08_S1(CO)_within_limits'. 

In [40]:
test26 = air_data.check_Null('NMHC(GT)').check_within_limits('PT08_S1(CO)', lower=900, upper=1300)
test26.df.orderBy(F.rand()).show(25)

+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+----------------+-------------------------+
|observation|      Date|               Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|T_within_limits|Date_value_in_list|NOx(GT)_is_Null|Date_is_Null|NMHC(GT)_is_Null|PT08_S1(CO)_within_limits|
+-----------+----------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+------------------+---------------+------------+----------------+-------------------------+
|       8422| 2/24/2005|2026-03-22 16:00:00|   3.0|       1274|    NULL|    13.3|         1094|    507|         537|    267|        1364|       1428| 7.6|70.8|0.7399|           t

*Result:* Success!

We can chain validation methods and also execute any validation or summarization method.

### Read in same data set using `pandas`

(Note that this is _not_ done using `pandas-on-spark`)

First, we will create a `pandas` DataFrame using the csv file. Then we will execute similar data cleaning methods to those used with the `from_csv()` method before executing our test.

In [41]:
# Create pandas DataFrame from CSV
air_panda = pd.read_csv('air.csv')
air_panda

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9352,9352,4/4/2005,10:00:00,3.1,1314,-200,13.5,1101,472,539,190,1374,1729,21.9,29.3,0.7568
9353,9353,4/4/2005,11:00:00,2.4,1163,-200,11.4,1027,353,604,179,1264,1269,24.3,23.7,0.7119
9354,9354,4/4/2005,12:00:00,2.4,1142,-200,12.4,1063,293,603,175,1241,1092,26.9,18.3,0.6406
9355,9355,4/4/2005,13:00:00,2.1,1003,-200,9.5,961,235,702,156,1041,770,28.3,13.5,0.5139


In [42]:
# Rename columns for convenience
air_panda = air_panda.rename(columns={'Unnamed: 0': 'observation',
                                      'PT08.S1(CO)': 'PT08_S1(CO)',
                                      'PT08.S2(NMHC)': 'PT08_S2(NMHC)',
                                      'PT08.S3(NOx)': 'PT08_S3(NOx)',
                                      'PT08.S4(NO2)': 'PT08_S4(NO2)',
                                      'PT08.S5(O3)': 'PT08_S5(O3)'})

# Change Time column to be string-type
air_panda['Time'] = air_panda['Time'].astype('str')

# Replace missing values of -200 with NaN
import numpy as np
air_panda = air_panda.replace(-200, np.NaN)

air_panda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9357 entries, 0 to 9356
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   observation    9357 non-null   int64  
 1   Date           9357 non-null   object 
 2   Time           9357 non-null   object 
 3   CO(GT)         7674 non-null   float64
 4   PT08_S1(CO)    8991 non-null   float64
 5   NMHC(GT)       914 non-null    float64
 6   C6H6(GT)       8991 non-null   float64
 7   PT08_S2(NMHC)  8991 non-null   float64
 8   NOx(GT)        7718 non-null   float64
 9   PT08_S3(NOx)   8991 non-null   float64
 10  NO2(GT)        7715 non-null   float64
 11  PT08_S4(NO2)   8991 non-null   float64
 12  PT08_S5(O3)    8991 non-null   float64
 13  T              8991 non-null   float64
 14  RH             8991 non-null   float64
 15  AH             8991 non-null   float64
dtypes: float64(13), int64(1), object(2)
memory usage: 1.1+ MB


In [43]:
air_panda.describe()

,observation,CO(GT),PT08_S1(CO),NMHC(GT),C6H6(GT),PT08_S2(NMHC),NOx(GT),PT08_S3(NOx),NO2(GT),PT08_S4(NO2),PT08_S5(O3),T,RH,AH
count,9357.000000,7674.000000,8991.000000,914.000000,8991.000000,8991.000000,7718.000000,8991.000000,7715.000000,8991.000000,8991.000000,8991.000000,8991.000000,8991.000000
mean,4678.000000,2.152750,1099.833166,218.811816,10.083105,939.153376,246.896735,835.493605,113.091251,1456.264598,1022.906128,18.317829,49.234201,1.025530
std,2701.277568,1.453252,217.080037,204.459921,7.449820,266.831429,212.979168,256.817320,48.370108,346.206794,398.484288,8.832116,17.316892,0.403813
min,0.000000,0.100000,647.000000,7.000000,0.100000,383.000000,2.000000,322.000000,2.000000,551.000000,221.000000,-1.900000,9.200000,0.184700
25%,2339.000000,1.100000,937.000000,67.000000,4.400000,734.500000,98.000000,658.000000,78.000000,1227.000000,731.500000,11.800000,35.800000,0.736800
50%,4678.000000,1.800000,1063.000000,150.000000,8.200000,909.000000,180.000000,806.000000,109.000000,1463.000000,963.000000,17.800000,49.600000,0.995400
75%,7017.000000,2.900000,1231.000000,297.000000,14.000000,1116.000000,326.000000,969.500000,142.000000,1674.000000,1273.500000,24.400000,62.500000,1.313700
max,9356.000000,11.900000,2040.000000,1189.000000,63.700000,2214.000000,1479.000000,2683.000000,340.000000,2775.000000,2523.000000,44.600000,88.700000,2.231000


Now we will create another instance of the `SparkDataCheck` class starting with the `pandas` data frame instead of reading the csv directly into the instance. 

In [45]:
# Create another test session for the air quality data
air_session2 = SparkSession.builder.appName('my_app2')\
                    .config("spark.sql.ansi.enabled", "false")\
                    .getOrCreate()
air_data2 = project2_script.SparkDataCheck.from_pandas(air_session2, air_panda)
air_data2.df.show(1)

+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|observation|     Date|    Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|
+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|          0|3/10/2004|18:00:00|   2.6|     1360.0|   150.0|    11.9|       1046.0|  166.0|      1056.0|  113.0|      1692.0|     1268.0|13.6|48.9|0.7578|
+-----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
only showing top 1 row


In [46]:
air_data2.df.printSchema()

root
 |-- observation: long (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- CO(GT): double (nullable = true)
 |-- PT08_S1(CO): double (nullable = true)
 |-- NMHC(GT): double (nullable = true)
 |-- C6H6(GT): double (nullable = true)
 |-- PT08_S2(NMHC): double (nullable = true)
 |-- NOx(GT): double (nullable = true)
 |-- PT08_S3(NOx): double (nullable = true)
 |-- NO2(GT): double (nullable = true)
 |-- PT08_S4(NO2): double (nullable = true)
 |-- PT08_S5(O3): double (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)



### Provide an example method call

Here, we will just show use of one method call since we previously demonstrated the ability to use each method. 

To maximize value, we will chain two validation methods together (similar to Test 26). 

**Test 27:** Chain two validation methods. 

**Expected behavior:** Successful execution. New Boolean columns will be added for 'NMHC(GT)_is_Null' and 'PT08_S1(CO)_within_limits'. 

In [47]:
test27 = air_data2.check_Null('NMHC(GT)').check_within_limits('PT08_S1(CO)', lower=900, upper=1300)
test27.df.orderBy(F.rand()).show(25)

+-----------+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------------+-------------------------+
|observation|      Date|    Time|CO(GT)|PT08_S1(CO)|NMHC(GT)|C6H6(GT)|PT08_S2(NMHC)|NOx(GT)|PT08_S3(NOx)|NO2(GT)|PT08_S4(NO2)|PT08_S5(O3)|   T|  RH|    AH|NMHC(GT)_is_Null|PT08_S1(CO)_within_limits|
+-----------+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------------+-------------------------+
|        104| 3/15/2004| 2:00:00|   1.8|     1224.0|    66.0|     7.0|        855.0|  108.0|       998.0|   88.0|      1566.0|     1149.0|13.4|61.3|0.9361|           false|                     true|
|       1257|  5/2/2004| 3:00:00|   1.6|     1000.0|    NULL|     6.6|        838.0|   NULL|       910.0|   NULL|      1580.0|      942.0|14.8|67.6|1.1352|            true|                     true|
|    

*Results:* Success! The additional columns showed up, and the values make sense. 

This shows we can equivalently add data from a csv directly into the class, or from csv to `pandas` to the class. 

## Part II - Basic spark analysis on NFL data

In this part, we will use Spark SQL and `pandas-on-Spark` to do some basic analysis on some NFL data.  The data was directly sourced from a csv file provided by the course instructor. 

The dataset includes 53 columns and 128,873 rows with statistics on NFL players from 1999 to 2023. The data contains null values (blank csv values converted to NULL in the Spark SQL DataFrame).

The first section will be completed solely using `pandas-on-Spark`

### `pandas-on-Spark`

#### Step 1: Read in the weekly NFL data from the csv

The csv file has been uploaded to the JupyterHub folder containing this notebook.

In [52]:
# Ensure pandas-on-spark environment is set up
import pandas as pd
import numpy as np
import pyspark.pandas as ps
ps.set_option('compute.fail_on_ansi_mode', False)

In [57]:
# Create pandas DataFrame and use it to create pandas-on-spark DataFrame
pdf = pd.read_csv('weekly_nfl_data.csv', delimiter = ",")
psdf = ps.from_pandas(pdf)

#### Step 2: Check out the first 5 rows of the DataFrame

We will do this by simply using the head() method

In [58]:
psdf.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


The DataFrame appears to have loaded successfully. The first few rows are all for the same player, but over different weeks of the 1999 season. Some values are missing because they apparently weren't recorded for some of those earliest years. 

#### Step 3: Report all of the column names

We could do this by calling `psdf.columns`, but I will use the `info()` method so we also get information on the column data types and number of non-null values. As anothe benefit, it's easier to read in this tabular format than a raw list.

In [61]:
psdf.info()

<class 'pyspark.pandas.frame.DataFrame'>
Index: 128873 entries, 0 to 128872
Data columns (total 53 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   player_id                    128873 non-null  object 
 1   player_name                  61493 non-null   object 
 2   player_display_name          128870 non-null  object 
 3   position                     128801 non-null  object 
 4   position_group               128801 non-null  object 
 5   headshot_url                 69819 non-null   object 
 6   recent_team                  128873 non-null  object 
 7   season                       128873 non-null  int64  
 8   week                         128873 non-null  int64  
 9   season_type                  128873 non-null  object 
 10  opponent_team                128873 non-null  object 
 11  completions                  128873 non-null  int64  
 12  attempts                     128873 non-null  int64  
 13  p

Each column is listed above. Many of them have the same non-null count (equal to the number of observations in the dataset), but several have varying numbers of missing values. Most of the columns are numeric (float64 or int64) but a few are objects (likely strings). 

#### Step 4: Look at QB stats for the seasons 2005 to 2023 (inclusive).

This step will have multiple sub-steps, as described below. 

##### Step 4.1: 

Subset the rows of the data to only include the position "QB", the regular season ("REG"), and season in the range noted above.

In [67]:
# Subset by position (QB), season_type (REG), and season (2005-2023)
subset1 = psdf[(psdf['position'] == 'QB') & 
               (psdf['season_type'] == 'REG') &
               (psdf['season'].between(2005, 2023, inclusive = 'both'))
              ]     
                   
subset1.tail()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
128850,00-0039163,C.Stroud,C.J. Stroud,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,HOU,2023,18,REG,IND,20,26,264.0,2,0.0,2.0,18.0,0,0,261.0,115.0,10.0,12.062858,0,1.011494,0.248064,3,20.0,0,0.0,0.0,2.0,2.007729,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,20.56,20.56
128853,00-0039164,A.Richardson,Anthony Richardson,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,IND,2023,1,REG,JAX,24,37,223.0,1,1.0,4.0,8.0,0,0,216.0,136.0,12.0,-4.387141,0,1.032407,-0.012689,10,40.0,1,0.0,0.0,3.0,-0.241498,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,20.92,20.92
128854,00-0039164,A.Richardson,Anthony Richardson,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,IND,2023,2,REG,HOU,6,10,56.0,0,0.0,0.0,-0.0,0,0,16.0,61.0,4.0,1.732920,0,3.500000,0.091884,3,35.0,2,0.0,0.0,2.0,3.513627,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,17.74,17.74
128855,00-0039164,A.Richardson,Anthony Richardson,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,IND,2023,4,REG,LA,11,25,200.0,2,0.0,2.0,4.0,0,0,311.0,56.0,10.0,5.105916,2,0.643087,0.002453,10,56.0,1,2.0,1.0,3.0,-2.365424,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,29.60,29.60
128856,00-0039164,A.Richardson,Anthony Richardson,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,IND,2023,5,REG,TEN,9,12,98.0,0,0.0,1.0,17.0,1,0,133.0,44.0,5.0,2.136096,0,0.736842,0.142546,2,5.0,0,0.0,0.0,0.0,-0.476602,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,4.42,4.42


In [77]:
subset1.shape

(11750, 53)

It appears that this subset was successfully executed

##### Step 4.2: 

Subset the columns to only include the `player_display~name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`.

Here, we will use the previous subset as a starting point. 

In [69]:
# Define columns of interest in a list
cols = ['player_display_name', 
        'season', 
        'week', 
        'completions', 
        'attempts', 
        'passing_yards', 
        'passing_tds', 
        'interceptions']

# Subset by this column list
subset2 = subset1[cols]
subset2.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


In [78]:
subset2.shape

(11750, 8)

The second subset was also successful! There are fewer observations in this subsetted dataset.

##### Step 4.3: 

For each `player_display_name` and `season` combination, find the *sum* and *mean* of each of the statistical quantities (the rest of the columns we chose above).

We will do this using the `groupby()` and `agg()` methods. 

First, let's see how many players made it into this subsetted DataFrame. 

In [79]:
subset2['player_display_name'].nunique()

302

There are 302 players and 19 seasons. Because each player didn't play that entire duration, we should have fewer than (302*19=5738) combinations for each statistic.

In [81]:
# Group and aggregate
view3 = subset2.groupby(['player_display_name', 'season'])\
               .agg(['sum', 'mean'])
view3.head()

week            completions            attempts            passing_yards             passing_tds           interceptions          
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                                   
Jeff Blake          2005     19   9.500000           8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000
Daunte Culpepper    2005     31   4.428571         139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286
Kerry Collins       2005    134   8.933333         302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667
Tony Banks          2005     17  17.000000          14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000
Charlie Batch       2005     35  11.666667          23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333

The table above clearly shows the desired statistics for each selected numeric column when grouped by the player_display_name and season. 

In [97]:
# Get DataFrame shape
view3.shape

(1456, 12)

Because there are 1456 rows, we know that each player appears multiple times. Let's check to make sure that, for a given player, multiple rows are included for each season played. We can do this by sorting the dataframe by the `player_display_name` first and the `season` second.

In [96]:
# Sort DataFrame
view3.sort_index(level=[0, 1], ascending = True, inplace = True)
view3.head(15)

week            completions            attempts            passing_yards             passing_tds           interceptions          
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean
player_display_name season                                                                                                                                   
A.J. Feeley         2006     29  14.500000          26  13.000000       38  19.000000         342.0  171.000000           3  1.500000           0.0  0.000000
                    2007     36  12.000000          59  19.666667      103  34.333333         681.0  227.000000           5  1.666667           8.0  2.666667
                    2011     29   7.250000          53  13.250000       97  24.250000         548.0  137.000000           1  0.250000           2.0  0.500000
AJ McCarron         2015     96  13.714286          79  11.285714      119  17.000000         854.0  122.000000           6  0.857143           2.0  0.285714
                    2017     29  14.500000           7   3.500000       14   7.000000          66.0   33.000000           0  0.000000           0.0  0.000000
                    2018     21  10.500000           1   0.500000        3   1.500000           8.0    4.000000           0  0.000000           0.0  0.000000
                    2019     28  14.000000          21  10.500000       37  18.500000         225.0  112.500000           0  0.000000           1.0  0.500000
                    2020     31  15.500000           1   0.500000        1   0.500000          20.0   10.000000           0  0.000000           0.0  0.000000
                    2023     32  16.000000           4   2.000000        5   2.500000          19.0    9.500000           0  0.000000           0.0  0.000000
Aaron Brooks        2005     95   7.307692         240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692
                    2006     85  10.625000         110  13.750000      192  24.000000        1105.0  138.125000           3  0.375000           8.0  1.000000
Aaron Rodgers       2005     37  12.333333           9   3.000000       16   5.333333          65.0   21.666667           0  0.000000           1.0  0.333333
                    2006     15   7.500000           6   3.000000       15   7.500000          46.0   23.000000           0  0.000000           0.0  0.000000
                    2007     23  11.500000          20  10.000000       28  14.000000         218.0  109.000000           1  0.500000           0.0  0.000000
                    2008    145   9.062500         341  21.312500      536  33.500000        4038.0  252.375000          28  1.750000          13.0  0.812500

Success! The aggregation worked correctly. 

##### Step 4.4: 

Create two new variables (by season/player combination):
+ `completion_percentage` = (sum of completions)/(sum of attempts)
+ `td_int_ratio` = (sum passing tds)/(sum interceptions)

In [100]:
# Create new variables
completion_percentage = view3[('completions', 'sum')] / \
                        view3[('attempts','sum')]
completion_percentage.head(15)

player_display_name  season
A.J. Feeley          2006      0.684211
                     2007      0.572816
                     2011      0.546392
AJ McCarron          2015      0.663866
                     2017      0.500000
                     2018      0.333333
                     2019      0.567568
                     2020      1.000000
                     2023      0.800000
Aaron Brooks         2005      0.556845
                     2006      0.572917
Aaron Rodgers        2005      0.562500
                     2006      0.400000
                     2007      0.714286
                     2008      0.636194
dtype: float64

In [99]:
# Create new variables (continued)
td_int_ratio = view3[('passing_tds', 'sum')] / \
               view3[('interceptions','sum')]
td_int_ratio.head(15)

player_display_name  season
A.J. Feeley          2006           inf
                     2007      0.625000
                     2011      0.500000
AJ McCarron          2015      3.000000
                     2017           NaN
                     2018           NaN
                     2019      0.000000
                     2020           NaN
                     2023           NaN
Aaron Brooks         2005      0.764706
                     2006      0.375000
Aaron Rodgers        2005      0.000000
                     2006           NaN
                     2007           inf
                     2008      2.153846
dtype: float64

By spot checking a few of the values, we see that the new variables were successfully created. Now that we trust the results, let's add them to the DataFrame. 

In [102]:
# Add variables to the view3 DataFrame
view3['completion_percentage'] = completion_percentage
view3['td_int_ratio'] = td_int_ratio
view3.head(15)

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
A.J. Feeley         2006     29  14.500000          26  13.000000       38  19.000000         342.0  171.000000           3  1.500000           0.0  0.000000              0.684211          inf
                    2007     36  12.000000          59  19.666667      103  34.333333         681.0  227.000000           5  1.666667           8.0  2.666667              0.572816     0.625000
                    2011     29   7.250000          53  13.250000       97  24.250000         548.0  137.000000           1  0.250000           2.0  0.500000              0.546392     0.500000
AJ McCarron         2015     96  13.714286          79  11.285714      119  17.000000         854.0  122.000000           6  0.857143           2.0  0.285714              0.663866     3.000000
                    2017     29  14.500000           7   3.500000       14   7.000000          66.0   33.000000           0  0.000000           0.0  0.000000              0.500000          NaN
                    2018     21  10.500000           1   0.500000        3   1.500000           8.0    4.000000           0  0.000000           0.0  0.000000              0.333333          NaN
                    2019     28  14.000000          21  10.500000       37  18.500000         225.0  112.500000           0  0.000000           1.0  0.500000              0.567568     0.000000
                    2020     31  15.500000           1   0.500000        1   0.500000          20.0   10.000000           0  0.000000           0.0  0.000000              1.000000          NaN
                    2023     32  16.000000           4   2.000000        5   2.500000          19.0    9.500000           0  0.000000           0.0  0.000000              0.800000          NaN
Aaron Brooks        2005     95   7.307692         240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692              0.556845     0.764706
                    2006     85  10.625000         110  13.750000      192  24.000000        1105.0  138.125000           3  0.375000           8.0  1.000000              0.572917     0.375000
Aaron Rodgers       2005     37  12.333333           9   3.000000       16   5.333333          65.0   21.666667           0  0.000000           1.0  0.333333              0.562500     0.000000
                    2006     15   7.500000           6   3.000000       15   7.500000          46.0   23.000000           0  0.000000           0.0  0.000000              0.400000          NaN
                    2007     23  11.500000          20  10.000000       28  14.000000         218.0  109.000000           1  0.500000           0.0  0.000000              0.714286          inf
                    2008    145   9.062500         341  21.312500      536  33.500000        4038.0  252.375000          28  1.750000          13.0  0.812500              0.636194     2.153846

Now that we trust the checkpoints along the way, we are ready to move onto the next stage. The object `view3` has our latest results. 

#### Step 5: Subset and sort the results

This step will have multiple sub-steps, as described below. 

##### Step 5.1: 
Subset the rows to only include player/season combinatino where the sum of attempts is at least 50

In [106]:
# Create new subset based on sum(attempts) >= 50
subset4 = view3[view3[('attempts', 'sum')] >= 50]
print(subset4.shape)
subset4.head()

(966, 14)


week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
A.J. Feeley         2007     36  12.000000          59  19.666667      103  34.333333         681.0  227.000000           5  1.666667           8.0  2.666667              0.572816     0.625000
                    2011     29   7.250000          53  13.250000       97  24.250000         548.0  137.000000           1  0.250000           2.0  0.500000              0.546392     0.500000
AJ McCarron         2015     96  13.714286          79  11.285714      119  17.000000         854.0  122.000000           6  0.857143           2.0  0.285714              0.663866     3.000000
Aaron Brooks        2005     95   7.307692         240  18.461538      431  33.153846        2882.0  221.692308          13  1.000000          17.0  1.307692              0.556845     0.764706
                    2006     85  10.625000         110  13.750000      192  24.000000        1105.0  138.125000           3  0.375000           8.0  1.000000              0.572917     0.375000

Of the 1456 rows, 966 remain after filtering by the sum of attempts for each player/season combination. 

##### Step 5.2: 
Sort the rows descending by `completion_percentage` and report the first 40 values!

In [110]:
# Sort by descending completion_percentage
sort5 = subset4.sort_values(by=['completion_percentage'], 
                   ascending = False)

# Report the first 40 values
sort5.head(40)

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
C.J. Beathard       2023     65  10.833333          40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Colt McCoy          2021     62   8.857143          74  10.571429       99  14.142857         740.0  105.714286           3  0.428571           1.0  0.142857              0.747475     3.000000
Matt Schaub         2019     52  10.400000          50  10.000000       67  13.400000         580.0  116.000000           3  0.600000           1.0  0.200000              0.746269     3.000000
Drew Brees          2018    130   8.666667         364  24.266667      489  32.600000        3992.0  266.133333          32  2.133333           5.0  0.333333              0.744376     6.400000
                    2019    119  10.818182         281  25.545455      378  34.363636        2979.0  270.818182          27  2.454545           4.0  0.363636              0.743386     6.750000
Mason Rudolph       2023     66  16.500000          55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Taysom Hill         2020    147   9.187500          88   5.500000      121   7.562500         928.0   58.000000           4  0.250000           2.0  0.125000              0.727273     2.000000
Nick Foles          2018     51  10.200000         141  28.200000      195  39.000000        1413.0  282.600000           7  1.400000           4.0  0.800000              0.723077     1.750000
Drew Brees          2017    148   9.250000         386  24.125000      536  33.500000        4334.0  270.875000          23  1.437500           8.0  0.500000              0.720149     2.875000
Sam Bradford        2016    146   9.733333         395  26.333333      552  36.800000        3877.0  258.466667          20  1.333333           5.0  0.333333              0.715580     4.000000
Drew Brees          2011    142   8.875000         471  29.437500      660  41.250000        5535.0  345.937500          46  2.875000          14.0  0.875000              0.713636     3.285714
Colt McCoy          2014     57  11.400000          91  18.200000      128  25.600000        1057.0  211.400000           4  0.800000           3.0  0.600000              0.710938     1.333333
Aaron Rodgers       2020    148   9.250000         372  23.250000      526  32.875000        4299.0  268.687500          48  3.000000           5.0  0.312500              0.707224     9.600000
Bailey Zappe        2022     22   5.500000          65  16.250000       92  23.000000         781.0  195.250000           5  1.250000           3.0  0.750000              0.706522     1.666667
Drew Brees          2009    131   8.733333         363  24.200000      514  34.266667        4388.0  292.533333          34  2.266667          11.0  0.733333              0.706226     3.090909
                    2020     97   8.083333         275  22.916667      390  32.500000        2942.0  245.166667          24  2.000000           6.0  0.500000              0.705128     4.000000
Joe Burrow          2021    143   8.937500         366  22.875000      520  32.500000        4611.0  288.187500          34  2.125000          14.0  0.875000              0.703846     2.428571
Derek Carr          2019    147   9.187500         361  22.562500      513  32.062500        4054.0  253.375000          21  1.312500           8.0  0.500000          

The `completion_percentage` column is now sorted in descending order, and the first 40 rows are displayed. 

##### Step 5.3: 
Sort the rows descending by `td_int_ratio` and report the first 40 values!

In [112]:
# Sort by descending td_int_ratio
sort6 = subset4.sort_values(by=['td_int_ratio'], 
                   ascending = False)

# Report the first 40 values
sort6.head(40)

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
C.J. Beathard       2023     65  10.833333          40   6.666667       53   8.833333         349.0   58.166667           1  0.166667           0.0  0.000000              0.754717          inf
Mason Rudolph       2023     66  16.500000          55  13.750000       74  18.500000         719.0  179.750000           3  0.750000           0.0  0.000000              0.743243          inf
Jimmy Garoppolo     2016     49   8.166667          43   7.166667       63  10.500000         502.0   83.666667           4  0.666667           0.0  0.000000              0.682540          inf
Derek Anderson      2014     30   6.000000          65  13.000000       97  19.400000         701.0  140.200000           5  1.000000           0.0  0.000000              0.670103          inf
Brian Hoyer         2016     27   4.500000         134  22.333333      200  33.333333        1445.0  240.833333           6  1.000000           0.0  0.000000              0.670000          inf
Nick Foles          2016     17   8.500000          36  18.000000       55  27.500000         410.0  205.000000           3  1.500000           0.0  0.000000              0.654545          inf
Matt Moore          2019     54   9.000000          59   9.833333       91  15.166667         659.0  109.833333           4  0.666667           0.0  0.000000              0.648352          inf
Todd Collins        2007     62  15.500000          67  16.750000      105  26.250000         888.0  222.000000           5  1.250000           0.0  0.000000              0.638095          inf
Desmond Ridder      2022     66  16.500000          73  18.250000      115  28.750000         708.0  177.000000           2  0.500000           0.0  0.000000              0.634783          inf
C.J. Beathard       2020     67  11.166667          66  11.000000      104  17.333333         787.0  131.166667           6  1.000000           0.0  0.000000              0.634615          inf
Andy Dalton         2023      8   4.000000          34  17.000000       58  29.000000         361.0  180.500000           2  1.000000           0.0  0.000000              0.586207          inf
Charlie Batch       2006     71  10.142857          30   4.285714       52   7.428571         477.0   68.142857           5  0.714286           0.0  0.000000              0.576923          inf
Troy Smith          2007     62  15.500000          40  10.000000       76  19.000000         452.0  113.000000           2  0.500000           0.0  0.000000              0.526316          inf
Matt Schaub         2005     65   9.285714          33   4.714286       64   9.142857         495.0   70.714286           4  0.571429           0.0  0.000000              0.515625          inf
Jake Locker         2011     51  10.200000          34   6.800000       66  13.200000         542.0  108.400000           4  0.800000           0.0  0.000000              0.515152          inf
Tom Brady           2016    134  11.166667         291  24.250000      432  36.000000        3554.0  296.166667          28  2.333333           2.0  0.166667              0.673611    14.000000
Nick Foles          2013    129   9.923077         203  15.615385      317  24.384615        2891.0  222.384615          27  2.076923           2.0  0.153846              0.640379    13.500000
Josh McCown         2013     92  11.500000         149  18.625000      224  28.000000        1829.0  228.625000          13  1.625000           1.0  0.125000          

The `td_int_ratio` column is now sorted in descending order, and the first 40 rows are displayed. We see that the first several rows have a `td_int_ratio` of "inf" (meaning infinity) because that player threw passing touchdowns but no interceptions in those seasons (any positive number divided by zero is infinity). We will later compare how the Spark SQL Data treats this operation. 